# Carelon App — Setup OAuth Scopes & Revoke Consent

Run this notebook once after creating the app (or after any scope change).
- Sets `user_api_scopes` so X-Forwarded-Access-Token has the `files` scope
- Revokes existing OAuth consent so next app access gets a fresh token with new scope

In [0]:
from databricks.sdk import WorkspaceClient
import requests, os

APP_NAME = 'carelon-app'
# 'all-apis' covers Jobs, Clusters, Files, SQL — everything admin needs
REQUIRED_SCOPES = ['all-apis']

# Use notebook's own auth context
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host = spark.conf.get("spark.databricks.workspaceUrl")

w = WorkspaceClient(host=f"https://{host}", token=token, auth_type='pat')

# Get current app config
app = w.apps.get(APP_NAME)
print(f"App: {app.name}")
print(f"Current effective scopes: {app.effective_user_api_scopes}")
print(f"OAuth integration ID: {app.oauth2_app_client_id}")

In [0]:
# Update app with required scopes
from databricks.sdk.service.apps import App
updated = w.apps.update(name=APP_NAME, app=App(name=APP_NAME, user_api_scopes=REQUIRED_SCOPES))
print(f"✅ Updated scopes: {updated.effective_user_api_scopes}")

In [0]:
# Revoke existing OAuth consent so next access gets fresh token with new scope
app_integration_id = app.oauth2_app_client_id

resp = requests.delete(
    f"https://{host}/api/2.0/oauth-app-integrations/{app_integration_id}/user-consent/me",
    headers={"Authorization": f"Bearer {token}"}
)

if resp.status_code == 200:
    print("✅ Consent revoked. Open the app in an incognito window — you'll be prompted to re-authorize with the 'files' scope.")
elif resp.status_code == 404:
    print("ℹ️ No existing consent found (already revoked or never consented). Just access the app URL to consent.")
else:
    print(f"⚠️ Status {resp.status_code}: {resp.text}")

## Next Steps
1. Open the app URL in an **incognito/private window**: https://carelon-app-7474651935606803.aws.databricksapps.com
2. You should see a **consent prompt** — accept it
3. The Browse button on the Upload page should now list Volume folders